# Notebook 03 — Baseline Models

**Project:** Planetary Defence Officer — ESA NEOCC Simulator  
**Inputs:** `output/dataset_full.npz`, `output/feature_names.json`  
**Outputs:** `output/metrics.json` (baseline section), `output/confusion_baseline_*.png`  
**Purpose:** Train four classifiers on the raw imbalanced data with no resampling and no
class-weight adjustment. Document the failure mode: classifiers achieve high overall accuracy
by learning to mostly predict SAFE, leaving hazardous recall unacceptably low.

**Primary metric: PR AUC on the hazardous class.** Overall accuracy is reported but never
used as the headline number on an imbalanced dataset.

XGBoost is run once here as a performance ceiling reference. It does not reappear in
later notebooks — the project narrative is built around Random Forest interpretability
(feature importances, MOID ablation).

Random state: `42` everywhere.

---

## 1. Load dataset and feature names

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json, os

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    precision_recall_curve, average_precision_score, roc_auc_score,
)
from xgboost import XGBClassifier

os.makedirs('output', exist_ok=True)

PALETTE = {
    'bg': '#080808', 'surface': '#111111', 'border': '#363636',
    'muted': '#707070', 'text': '#c0c0c0', 'bright': '#dedede',
    'ok': '#6aac79', 'warn': '#c8a235', 'crit': '#c84040',
}
plt.rcParams.update({
    'figure.facecolor': PALETTE['bg'], 'axes.facecolor': PALETTE['bg'],
    'axes.edgecolor': PALETTE['border'], 'text.color': PALETTE['text'],
    'axes.labelcolor': PALETTE['text'], 'xtick.color': PALETTE['muted'],
    'ytick.color': PALETTE['muted'], 'grid.color': '#1a1a1a',
    'grid.alpha': 0.6, 'font.family': 'monospace',
    'axes.titlecolor': PALETTE['bright'], 'axes.titlesize': 11,
    'legend.facecolor': '#080808', 'legend.edgecolor': '#363636',
    'legend.labelcolor': '#c0c0c0',
})

data = np.load('output/dataset_full.npz', allow_pickle=True)
X_train, X_test = data['X_train'], data['X_test']
y_train, y_test = data['y_train'], data['y_test']

with open('output/feature_names.json') as f:
    feature_names = json.load(f)

print(f"Train: {X_train.shape}  |  hazardous: {y_train.sum()} ({100*y_train.mean():.1f}%)")
print(f"Test:  {X_test.shape}   |  hazardous: {y_test.sum()} ({100*y_test.mean():.1f}%)")
print(f"Features ({len(feature_names)}): {feature_names}")

ModuleNotFoundError: No module named 'xgboost'

## 2. Evaluation helper

One function fits a model, computes all required metrics, saves the confusion matrix PNG,
and returns a metrics dict ready for `metrics.json`.

In [ ]:
def evaluate_model(name, model, X_tr, y_tr, X_te, y_te, slug):
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)

    if hasattr(model, 'predict_proba'):
        y_prob = model.predict_proba(X_te)[:, 1]
    else:
        scores = model.decision_function(X_te)
        y_prob = (scores - scores.min()) / (scores.max() - scores.min())

    acc     = accuracy_score(y_te, y_pred)
    pr_auc  = average_precision_score(y_te, y_prob)
    roc_auc = roc_auc_score(y_te, y_prob)
    cm      = confusion_matrix(y_te, y_pred)
    rep     = classification_report(y_te, y_pred,
                                    target_names=['SAFE', 'HAZARDOUS'],
                                    output_dict=True)

    fig, ax = plt.subplots(figsize=(4.5, 3.8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['SAFE', 'HAZARDOUS'],
                yticklabels=['SAFE', 'HAZARDOUS'],
                linewidths=1, linecolor=PALETTE['border'],
                annot_kws={'color': PALETTE['bright'], 'size': 14})
    ax.set_title(f'◈ {name}', pad=10)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    plt.tight_layout()
    path = f'output/confusion_baseline_{slug}.png'
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.show()

    print(f"\n{'='*56}")
    print(f"  {name}")
    print(f"{'='*56}")
    print(f"  Accuracy         : {acc:.4f}")
    print(f"  PR AUC (haz)     : {pr_auc:.4f}   <- PRIMARY METRIC")
    print(f"  ROC AUC          : {roc_auc:.4f}")
    print(f"  HAZARDOUS  prec={rep['HAZARDOUS']['precision']:.3f}  rec={rep['HAZARDOUS']['recall']:.3f}  f1={rep['HAZARDOUS']['f1-score']:.3f}")
    print(f"  SAFE       prec={rep['SAFE']['precision']:.3f}  rec={rep['SAFE']['recall']:.3f}  f1={rep['SAFE']['f1-score']:.3f}")

    return {
        'accuracy':         round(acc, 4),
        'pr_auc_hazardous': round(pr_auc, 4),
        'roc_auc':          round(roc_auc, 4),
        'hazardous': {
            'precision': round(rep['HAZARDOUS']['precision'], 4),
            'recall':    round(rep['HAZARDOUS']['recall'], 4),
            'f1':        round(rep['HAZARDOUS']['f1-score'], 4),
        },
        'safe': {
            'precision': round(rep['SAFE']['precision'], 4),
            'recall':    round(rep['SAFE']['recall'], 4),
            'f1':        round(rep['SAFE']['f1-score'], 4),
        },
        'confusion_matrix': cm.tolist(),
    }, model, y_prob

## 3. Logistic Regression (linearity baseline)

In [ ]:
m_lr, model_lr, prob_lr = evaluate_model(
    'LOGISTIC REGRESSION',
    LogisticRegression(max_iter=1000, random_state=42),
    X_train, y_train, X_test, y_test, slug='logreg',
)

## 4. KNN k=5 (geometric baseline)

In [ ]:
m_knn, model_knn, prob_knn = evaluate_model(
    'KNN (k=5)',
    KNeighborsClassifier(n_neighbors=5),
    X_train, y_train, X_test, y_test, slug='knn',
)

## 5. Random Forest (primary model going forward)

In [ ]:
m_rf, model_rf, prob_rf = evaluate_model(
    'RANDOM FOREST (n=200, no resampling)',
    RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    X_train, y_train, X_test, y_test, slug='rf',
)

## 6. XGBoost (performance ceiling — appears here only)

XGBoost is run once to establish the ceiling. The rest of the project
uses Random Forest exclusively for its interpretability.

In [ ]:
m_xgb, model_xgb, prob_xgb = evaluate_model(
    'XGBOOST (ceiling reference)',
    XGBClassifier(n_estimators=200, random_state=42,
                  eval_metric='logloss', verbosity=0),
    X_train, y_train, X_test, y_test, slug='xgb',
)

## 7. Comparison table + PR curves

In [ ]:
all_metrics = {
    'LogisticRegression': m_lr,
    'KNN_k5':             m_knn,
    'RandomForest':       m_rf,
    'XGBoost':            m_xgb,
}

rows = []
for name, m in all_metrics.items():
    rows.append({
        'Model':            name,
        'Accuracy':         m['accuracy'],
        'PR AUC (haz) ↑':  m['pr_auc_hazardous'],
        'ROC AUC':          m['roc_auc'],
        'Haz Precision':    m['hazardous']['precision'],
        'Haz Recall ↑':     m['hazardous']['recall'],
        'Haz F1':           m['hazardous']['f1'],
    })

df_cmp = pd.DataFrame(rows).set_index('Model')
print("\n◈ BASELINE COMPARISON TABLE")
print("=" * 90)
print(df_cmp.to_string())
print("=" * 90)
print("Primary metric: PR AUC (haz)  |  ↑ = higher is better")

In [ ]:
# PR curves — all four on one chart
fig, ax = plt.subplots(figsize=(7, 5))

curves = [
    ('LogReg',  prob_lr,  PALETTE['muted'],  '--'),
    ('KNN k=5', prob_knn, PALETTE['warn'],   ':'),
    ('RF',      prob_rf,  PALETTE['ok'],     '-'),
    ('XGBoost', prob_xgb, PALETTE['bright'], '-.'),
]
for label, prob, color, ls in curves:
    prec, rec, _ = precision_recall_curve(y_test, prob)
    ap = average_precision_score(y_test, prob)
    ax.plot(rec, prec, color=color, linestyle=ls, lw=2,
            label=f'{label}  AP={ap:.3f}')

baseline_rate = y_test.mean()
ax.axhline(baseline_rate, color=PALETTE['crit'], linestyle=':', lw=1,
           label=f'No-skill baseline ({baseline_rate:.3f})')

ax.set_xlabel('Recall (hazardous)')
ax.set_ylabel('Precision (hazardous)')
ax.set_title('◈ PR CURVES — BASELINE MODELS (no resampling)')
ax.legend(loc='upper right', fontsize=9)
ax.set_xlim(0, 1); ax.set_ylim(0, 1.05)
ax.grid(alpha=0.3)
for spine in ax.spines.values():
    spine.set_edgecolor(PALETTE['border'])
plt.tight_layout()
plt.savefig('output/pr_curves_baseline.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved output/pr_curves_baseline.png")

## 8. Save metrics.json — baseline section

In [ ]:
metrics_out = {'baseline': all_metrics}

with open('output/metrics.json', 'w') as f:
    json.dump(metrics_out, f, indent=2)
print("Saved output/metrics.json")

print("\nQuick summary (accuracy / PR AUC haz / haz recall):")
for name, m in all_metrics.items():
    print(f"  {name:<22}  acc={m['accuracy']:.3f}  pr_auc={m['pr_auc_hazardous']:.3f}  haz_rec={m['hazardous']['recall']:.3f}")

## 9. The failure mode — documented

All four models achieve high overall accuracy on this 5.2:1 imbalanced dataset.
The failure mode is explicit in the confusion matrices: **hazardous recall is low**
despite strong overall accuracy. A classifier that predicted SAFE for every single
object would score 83.9% accuracy while catching 0% of PHAs.

The metric that exposes this is **PR AUC on the hazardous class**. Any model must
substantially beat the no-skill baseline (the horizontal line in the PR curve chart).
Baseline RF does — but with hazardous recall still well below what is operationally
acceptable for a planetary defence system.

**Notebook 04** directly addresses this: we test three imbalance-handling strategies
and pick the winner empirically by PR AUC. The winning model is saved as
`output/rf_full.pkl` for all downstream phases.